# Beta Scan VAE Training - Disentanglement Optimization
Systematic beta scanning to find optimal disentanglement:
- Start with beta=1e-5
- Multiply by 3 each iteration
- Load weights from previous beta run
- Continue until KL < 10 at early stopping

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import time
from tqdm.auto import tqdm
import nibabel as nib
import json

# Runtime reload
import importlib
import models.autoencoder.bottleneck_models
importlib.reload(models.autoencoder.bottleneck_models)

from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.vae.vae_bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckVAE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Define Weighted VAE Loss Function

In [ ]:
class WeightedVAELoss(nn.Module):
    """
    Weighted VAE Loss with Luca's improvements:
    - Fixes 3D Axis Mismatch using permute(2, 1, 0)
    - Removes weight normalization
    - Uses 0.1 padding for background
    - Implements Beta Warmup from 0 to beta_final
    """
    def __init__(self, weight_mask_path, target_shape=None, beta_final=0.0005, warmup_steps=5000):
        super().__init__()
        
        # Load the Mask
        weight_nii = nib.load(weight_mask_path)
        weight_data = torch.from_numpy(weight_nii.get_fdata()).float()
        
        # Fix 3D axis swap
        weight_data = weight_data.permute(2, 1, 0)
        
        # Add batch/channel dims: [1, 1, D, H, W]
        self.full_mask = weight_data.unsqueeze(0).unsqueeze(0)
        
        # No normalization (per Luca)
        
        # Register as buffer
        self.register_buffer('weight_mask', self.full_mask)
        
        self.beta_final = beta_final
        self.warmup_steps = warmup_steps
        self.current_step = 0
        
        print(f"✅ Loss Initialized. Mask Permuted (2,1,0). Shape: {self.full_mask.shape}")
        print(f"   Beta: 0 → {beta_final} over {warmup_steps} steps")
        
    def smart_adjust_mask(self, mask, target_shape):
        """
        Adjusts mask to match the input batch size (D, H, W).
        Uses padding value 0.1 for background.
        """
        _, _, D_cur, H_cur, W_cur = mask.shape
        D_tgt, H_tgt, W_tgt = target_shape
        
        # PAD if smaller
        pad_d = max(0, D_tgt - D_cur)
        pad_h = max(0, H_tgt - H_cur)
        pad_w = max(0, W_tgt - W_cur)
        
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            mask = F.pad(mask, (pad_w//2, pad_w-pad_w//2, 
                                pad_h//2, pad_h-pad_h//2, 
                                pad_d//2, pad_d-pad_d//2), 
                         mode='constant', value=0.1)

        # CROP if larger
        _, _, D_new, H_new, W_new = mask.shape
        d_start = (D_new - D_tgt) // 2
        h_start = (H_new - H_tgt) // 2
        w_start = (W_new - W_tgt) // 2
        
        return mask[:, :, d_start:d_start+D_tgt, h_start:h_start+H_tgt, w_start:w_start+W_tgt]
    
    def forward(self, recon_x, x, mu, log_var):
        # Ensure mask is on correct device
        if self.weight_mask.device != recon_x.device:
            self.weight_mask = self.weight_mask.to(recon_x.device)
            
        current_target_shape = recon_x.shape[2:] 
        
        # Adjust mask to current batch dimensions if needed
        if self.weight_mask.shape[2:] != current_target_shape:
            adjusted_mask = self.smart_adjust_mask(self.weight_mask, current_target_shape)
        else:
            adjusted_mask = self.weight_mask
            
        # Weighted Reconstruction Loss
        squared_error = (recon_x - x) ** 2
        weighted_error = squared_error * adjusted_mask
        recon_loss = weighted_error.mean()
        
        # KL Divergence
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1)
        kl_loss = torch.mean(kl_loss)
        
        # Beta Warmup (cycling benefit with initial_beta=0)
        if self.current_step < self.warmup_steps:
            beta = self.beta_final * (self.current_step / self.warmup_steps)
        else:
            beta = self.beta_final
            
        self.current_step += 1
        
        # Total Loss
        total_loss = recon_loss + (beta * kl_loss)
        
        return total_loss, recon_loss, kl_loss, beta

## Data Setup

In [ ]:
# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 4
output_dir = "output/Experiments/BetaScanVAE"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)
display(df.head())

# Create dataloaders
print("Creating dataloaders...")
if __name__ == '__main__':
    train_loader, val_loader = create_dataloaders(
        df, batch_size=batch_size, train_split=0.8, 
        on_demand=True, mask_path=mask_path,
        num_workers=2
    )

print(f"Data preparation complete. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

## Beta Scan Configuration

In [ ]:
# Beta scan parameters
beta_values = [1e-5, 3e-5, 9e-5, 2.7e-4, 8.1e-4, 2.43e-3, 7.29e-3]
current_beta_index = 0  # Change this to continue from a specific beta
current_beta = beta_values[current_beta_index]

# Directory to store beta scan results
beta_scan_dir = os.path.join(output_dir, "beta_scan_results")
os.makedirs(beta_scan_dir, exist_ok=True)

# Metrics storage for all beta runs
beta_scan_results = {
    'beta': [],
    'final_train_recon': [],
    'final_train_kl': [],
    'final_val_recon': [],
    'final_val_kl': [],
    'final_beta_kl': [],
    'epochs_trained': []
}

# Try to load existing results
summary_path = os.path.join(beta_scan_dir, "beta_scan_summary.json")
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        beta_scan_results = json.load(f)
    print(f"✓ Loaded existing beta scan results: {len(beta_scan_results['beta'])} runs completed")

print(f"\nBeta scan configured: {beta_values}")
print(f"Starting with beta = {current_beta:.2e}")
print(f"Results directory: {beta_scan_dir}")

## Model Setup

In [ ]:
target_shape = (64, 128, 128)
latent_dim = 256

def build_vae_model(latent_dim=256):
    """Build BottleneckVAE model"""
    model = BottleneckVAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
        latent_dim=latent_dim
    )
    return model.to(device)

model = build_vae_model(latent_dim=latent_dim)

# Load weights from previous beta run if available
if current_beta_index > 0:
    prev_beta = beta_values[current_beta_index - 1]
    prev_checkpoint = os.path.join(beta_scan_dir, f"beta_{prev_beta:.2e}_best.pth")
    if os.path.exists(prev_checkpoint):
        model.load_state_dict(torch.load(prev_checkpoint))
        print(f"✓ Loaded weights from previous beta run: {prev_checkpoint}")
    else:
        print(f"⚠ Previous checkpoint not found: {prev_checkpoint}")
        print("Training from scratch...")
else:
    print("Starting first beta run from scratch")

num_params = count_trainable(model)
print(f"Model Parameters: {num_params / 1e6:.2f}M")

## Initialize Loss Function

In [ ]:
# Initialize loss function for current beta
criterion = WeightedVAELoss(
    weight_mask_path="data/masks/weightMatrix.nii",
    beta_final=current_beta,
    warmup_steps=5000
)
print(f"\nLoss initialized with beta_final = {current_beta:.2e}")

## Training Functions

In [ ]:
def train_vae_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    for batch_idx, batch in enumerate(pbar):
        data = batch['volume'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        recon, mu, log_var = model(data)
        
        # Compute loss
        loss, recon_loss, kl_loss, beta = criterion(recon, data, mu, log_var)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.6f}',
            'recon': f'{recon_loss.item():.6f}',
            'kl': f'{kl_loss.item():.6f}',
            'beta': f'{beta:.6f}'
        })
    
    avg_loss = total_loss / len(train_loader)
    avg_recon = total_recon / len(train_loader)
    avg_kl = total_kl / len(train_loader)
    
    return avg_loss, avg_recon, avg_kl

def validate_vae(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation", leave=False)
        for batch in pbar:
            data = batch['volume'].to(device)
            
            # Forward pass
            recon, mu, log_var = model(data)
            
            # Compute loss
            loss, recon_loss, kl_loss, beta = criterion(recon, data, mu, log_var)
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.6f}',
                'recon': f'{recon_loss.item():.6f}',
                'kl': f'{kl_loss.item():.6f}'
            })
    
    avg_loss = total_loss / len(val_loader)
    avg_recon = total_recon / len(val_loader)
    avg_kl = total_kl / len(val_loader)
    
    return avg_loss, avg_recon, avg_kl

## Training Loop

In [ ]:
# Training configuration
num_epochs = 150
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

# Training history
train_losses = []
val_losses = []
train_recon_losses = []
train_kl_losses = []
val_recon_losses = []
val_kl_losses = []

best_val_loss = float('inf')
patience_counter = 0
early_stopping_patience = 20

# Beta-specific checkpoint directory
model_name = f"beta_{current_beta:.2e}"
checkpoint_dir = os.path.join(beta_scan_dir, model_name)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Checkpoint directory: {checkpoint_dir}")

In [ ]:
if __name__ == '__main__':
    print(f"\n{'='*60}")
    print(f"Starting training for beta = {current_beta:.2e}")
    print(f"{'='*60}\n")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")
        
        # Train
        train_loss, train_recon, train_kl = train_vae_epoch(model, train_loader, optimizer, criterion, device)
        train_losses.append(train_loss)
        train_recon_losses.append(train_recon)
        train_kl_losses.append(train_kl)
        
        # Validate
        val_loss, val_recon, val_kl = validate_vae(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_recon_losses.append(val_recon)
        val_kl_losses.append(val_kl)
        
        print(f"Train Loss: {train_loss:.6f} (Recon: {train_recon:.6f}, KL: {train_kl:.2f})")
        print(f"Val Loss: {val_loss:.6f} (Recon: {val_recon:.6f}, KL: {val_kl:.2f})")
        print(f"Beta*KL: {current_beta * val_kl:.6f}")
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping and checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            best_model_path = os.path.join(beta_scan_dir, f"beta_{current_beta:.2e}_best.pth")
            torch.save(model.state_dict(), best_model_path)
            print(f"✓ Best model saved: {best_model_path}")
        else:
            patience_counter += 1
        
        # Save periodic checkpoint
        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f"epoch{epoch+1}.pth")
            torch.save(model.state_dict(), checkpoint_path)
            print(f"✓ Checkpoint saved: {checkpoint_path}")
        
        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    elapsed_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✓ Training completed in {elapsed_time:.1f}s ({elapsed_time/60:.1f} min)")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Final validation KL: {val_kl_losses[-1]:.2f}")
    print(f"{'='*60}\n")
    
    # Store results for this beta
    beta_scan_results['beta'].append(current_beta)
    beta_scan_results['final_train_recon'].append(train_recon_losses[-1])
    beta_scan_results['final_train_kl'].append(train_kl_losses[-1])
    beta_scan_results['final_val_recon'].append(val_recon_losses[-1])
    beta_scan_results['final_val_kl'].append(val_kl_losses[-1])
    beta_scan_results['final_beta_kl'].append(current_beta * val_kl_losses[-1])
    beta_scan_results['epochs_trained'].append(len(train_losses))
    
    # Save metrics for this beta run
    metrics_path = os.path.join(checkpoint_dir, "metrics.npz")
    np.savez(metrics_path,
             train_losses=train_losses,
             val_losses=val_losses,
             train_recon=train_recon_losses,
             train_kl=train_kl_losses,
             val_recon=val_recon_losses,
             val_kl=val_kl_losses,
             beta=current_beta)
    print(f"✓ Metrics saved: {metrics_path}")
    
    # Save beta scan summary
    with open(summary_path, 'w') as f:
        json.dump(beta_scan_results, f, indent=2)
    print(f"✓ Beta scan summary saved: {summary_path}")
    
    # Decision guidance
    print(f"\n{'='*60}")
    print("NEXT STEPS:")
    if val_kl_losses[-1] > 10:
        print(f"✓ KL={val_kl_losses[-1]:.2f} > 10: Continue to next beta")
        print(f"  Set current_beta_index = {current_beta_index + 1}")
        if current_beta_index + 1 < len(beta_values):
            print(f"  Next beta: {beta_values[current_beta_index + 1]:.2e}")
    else:
        print(f"✓ KL={val_kl_losses[-1]:.2f} < 10: Beta scan complete!")
        print("  Proceed to analysis and visualization")
    print(f"{'='*60}\n")

## Training Curves for Current Beta

In [ ]:
# Plot training curves for current beta
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

epochs = range(1, len(train_losses) + 1)

# Total Loss
axes[0, 0].plot(epochs, train_losses, label='Train', linewidth=2)
axes[0, 0].plot(epochs, val_losses, label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title(f'Total Loss (Beta={current_beta:.2e})', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Reconstruction Loss
axes[0, 1].plot(epochs, train_recon_losses, label='Train', linewidth=2)
axes[0, 1].plot(epochs, val_recon_losses, label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Reconstruction Loss')
axes[0, 1].set_title('Reconstruction Loss', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# KL Divergence
axes[1, 0].plot(epochs, train_kl_losses, label='Train', linewidth=2)
axes[1, 0].plot(epochs, val_kl_losses, label='Val', linewidth=2)
axes[1, 0].axhline(y=10, color='r', linestyle='--', label='KL=10 threshold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('KL Divergence')
axes[1, 0].set_title('KL Divergence', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Beta * KL
beta_kl_train = [current_beta * kl for kl in train_kl_losses]
beta_kl_val = [current_beta * kl for kl in val_kl_losses]
axes[1, 1].plot(epochs, beta_kl_train, label='Train', linewidth=2)
axes[1, 1].plot(epochs, beta_kl_val, label='Val', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Beta * KL')
axes[1, 1].set_title('Beta * KL Divergence', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(checkpoint_dir, f"training_curves_beta_{current_beta:.2e}.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Training curves saved: {plot_path}")

## Beta Scan Summary and Analysis

In [ ]:
# Load beta scan summary
summary_path = os.path.join(beta_scan_dir, "beta_scan_summary.json")
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        beta_scan_results = json.load(f)
    
    print("Beta Scan Results Summary:")
    print("="*80)
    print(f"{'Beta':<12} {'Val Recon':<12} {'Val KL':<10} {'Beta*KL':<12} {'Epochs':<8}")
    print("="*80)
    for i, beta in enumerate(beta_scan_results['beta']):
        print(f"{beta:<12.2e} {beta_scan_results['final_val_recon'][i]:<12.6f} "
              f"{beta_scan_results['final_val_kl'][i]:<10.2f} "
              f"{beta_scan_results['final_beta_kl'][i]:<12.6f} "
              f"{beta_scan_results['epochs_trained'][i]:<8}")
    print("="*80)
else:
    print("No beta scan summary found. Run training first.")

## Beta Scan Visualization

In [ ]:
# Plot beta scan results across all beta values
if len(beta_scan_results['beta']) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    betas = beta_scan_results['beta']
    
    # Plot 1: Reconstruction Error vs Beta
    axes[0, 0].plot(betas, beta_scan_results['final_train_recon'], 'o-', label='Train Recon', linewidth=2, markersize=8)
    axes[0, 0].plot(betas, beta_scan_results['final_val_recon'], 's-', label='Val Recon', linewidth=2, markersize=8)
    axes[0, 0].set_xlabel('Beta', fontsize=12)
    axes[0, 0].set_ylabel('Reconstruction Error', fontsize=12)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_title('Reconstruction Error vs Beta', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: KL Divergence vs Beta
    axes[0, 1].plot(betas, beta_scan_results['final_train_kl'], 'o-', label='Train KL', linewidth=2, markersize=8)
    axes[0, 1].plot(betas, beta_scan_results['final_val_kl'], 's-', label='Val KL', linewidth=2, markersize=8)
    axes[0, 1].axhline(y=10, color='r', linestyle='--', linewidth=2, label='KL=10 threshold')
    axes[0, 1].set_xlabel('Beta', fontsize=12)
    axes[0, 1].set_ylabel('KL Divergence', fontsize=12)
    axes[0, 1].set_xscale('log')
    axes[0, 1].set_title('KL Divergence vs Beta', fontsize=14, fontweight='bold')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Beta * KL vs Beta
    axes[1, 0].plot(betas, beta_scan_results['final_beta_kl'], 'o-', color='purple', linewidth=2, markersize=8)
    axes[1, 0].set_xlabel('Beta', fontsize=12)
    axes[1, 0].set_ylabel('Beta * KL', fontsize=12)
    axes[1, 0].set_xscale('log')
    axes[1, 0].set_title('Beta * KL vs Beta', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Epochs Trained vs Beta
    axes[1, 1].bar(range(len(betas)), beta_scan_results['epochs_trained'], color='teal', alpha=0.7)
    axes[1, 1].set_xlabel('Beta', fontsize=12)
    axes[1, 1].set_ylabel('Epochs Trained', fontsize=12)
    axes[1, 1].set_title('Training Duration per Beta', fontsize=14, fontweight='bold')
    axes[1, 1].set_xticks(range(len(betas)))
    axes[1, 1].set_xticklabels([f"{b:.2e}" for b in betas], rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plot_path = os.path.join(beta_scan_dir, "beta_scan_analysis.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Beta scan plot saved: {plot_path}")
else:
    print("Need at least 2 beta runs to plot comparison")

## Mask Alignment Verification

In [ ]:
# Verify mask alignment with brain image
print("Verifying mask alignment...")

# Load mask
mask_nii = nib.load("data/masks/weightMatrix.nii")
mask_data = torch.from_numpy(mask_nii.get_fdata()).float()
mask_data = mask_data.permute(2, 1, 0)  # Apply same permutation as in loss

# Load a sample brain
sample_batch = next(iter(val_loader))
sample_brain = sample_batch['volume'][0, 0].cpu().numpy()  # [D, H, W]

# Adjust mask to match brain shape
mask_adjusted = criterion.smart_adjust_mask(
    mask_data.unsqueeze(0).unsqueeze(0), 
    sample_brain.shape
)[0, 0].cpu().numpy()

# Plot overlay on multiple slices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

slice_positions = [
    ('Axial', 0, sample_brain.shape[0]//2),
    ('Coronal', 1, sample_brain.shape[1]//2),
    ('Sagittal', 2, sample_brain.shape[2]//2)
]

for idx, (plane, axis, pos) in enumerate(slice_positions):
    # Get slices
    if axis == 0:
        brain_slice = sample_brain[pos, :, :]
        mask_slice = mask_adjusted[pos, :, :]
    elif axis == 1:
        brain_slice = sample_brain[:, pos, :]
        mask_slice = mask_adjusted[:, pos, :]
    else:
        brain_slice = sample_brain[:, :, pos]
        mask_slice = mask_adjusted[:, :, pos]
    
    # Brain only
    axes[0, idx].imshow(brain_slice, cmap='gray')
    axes[0, idx].set_title(f'{plane} - Brain', fontsize=12, fontweight='bold')
    axes[0, idx].axis('off')
    
    # Overlay
    axes[1, idx].imshow(brain_slice, cmap='gray')
    axes[1, idx].imshow(mask_slice, cmap='Reds', alpha=0.3)
    axes[1, idx].set_title(f'{plane} - Brain + Mask Overlay', fontsize=12, fontweight='bold')
    axes[1, idx].axis('off')

plt.tight_layout()
overlay_path = os.path.join(output_dir, "mask_alignment_verification.png")
plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Mask alignment verification saved: {overlay_path}")
print("\nCheck if mask aligns properly with brain structures (no flipping)")

## Reconstruction Visualization

In [ ]:
# Load best model from current beta
best_model = build_vae_model(latent_dim=latent_dim)
best_model_path = os.path.join(beta_scan_dir, f"beta_{current_beta:.2e}_best.pth")
if os.path.exists(best_model_path):
    best_model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model from: {best_model_path}")
else:
    print("Best model not found, using current model")
    best_model = model

best_model.eval()

# Get sample data
num_samples = 3
sample_batch = next(iter(val_loader))
sample_data = sample_batch['volume'][:num_samples].to(device)

# Generate reconstructions
with torch.no_grad():
    recons, _, _ = best_model(sample_data)

# Visualization
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
if num_samples == 1:
    axes = axes.reshape(1, -1)

for i in range(num_samples):
    # Extract middle slices
    orig = sample_data[i, 0].cpu().numpy()
    recon = recons[i, 0].cpu().numpy()
    diff = np.abs(orig - recon)
    
    mid_slice = orig.shape[0] // 2
    
    # Original
    axes[i, 0].imshow(orig[mid_slice], cmap='gray')
    axes[i, 0].set_title('Original', fontweight='bold')
    axes[i, 0].axis('off')
    
    # Reconstruction
    axes[i, 1].imshow(recon[mid_slice], cmap='gray')
    axes[i, 1].set_title('Reconstruction', fontweight='bold')
    axes[i, 1].axis('off')
    
    # Difference
    axes[i, 2].imshow(diff[mid_slice], cmap='hot')
    axes[i, 2].set_title('Absolute Difference', fontweight='bold')
    axes[i, 2].axis('off')

plt.tight_layout()
recon_path = os.path.join(checkpoint_dir, f"reconstructions_beta_{current_beta:.2e}.png")
plt.savefig(recon_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Reconstructions saved: {recon_path}")